🥉 **Camada Bronze**

Criação do Banco de Dados:

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;

Criação das tabelas a partir de cada arquivo .csv do dataset e adição da coluna timestamp_ingestion em cada uma das tabelas:

In [0]:
base_path = "/Volumes/rocket_ecommerce/default/rocket_ecommerce/"
arquivos_tabelas = {
    "olist_customers_dataset.csv": "bronze.tb_customers",
    "olist_geolocation_dataset.csv": "bronze.tb_geolocalizacao",
    "olist_order_items_dataset.csv": "bronze.tb_order_items",
    "olist_order_payments_dataset.csv": "bronze.tb_order_payments",
    "olist_order_reviews_dataset.csv": "bronze.tb_order_reviews",
    "olist_orders_dataset.csv": "bronze.tb_orders",
    "olist_products_dataset.csv": "bronze.tb_products",
    "olist_sellers_dataset.csv": "bronze.tb_sellers",
    "product_category_name_translation.csv": "bronze.tb_product_category_name_translation"
}

from pyspark.sql.functions import current_timestamp

for arquivo, tabela in arquivos_tabelas.items():
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(base_path + arquivo)
    
    df = df.withColumn("timestamp_ingestion", current_timestamp())
    
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(tabela)

Listando todas as colunas de uma tabela para verificar se a coluna timestamp_ingestion foi adicionada: 

In [0]:
%sql
DESCRIBE bronze.tb_orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_status,string,null
order_purchase_timestamp,timestamp,null
order_approved_at,timestamp,null
order_delivered_carrier_date,timestamp,null
order_delivered_customer_date,timestamp,null
order_estimated_delivery_date,timestamp,null
timestamp_ingestion,timestamp,null


Descobrindo a data de pedido mais antiga e a mais recente para utilizar como base na criação da tabela bronze.tb_cotacao_dolar:

In [0]:
%sql
SELECT
    MIN(DATE(order_purchase_timestamp)) AS data_mais_antiga,
    MAX(DATE(order_purchase_timestamp)) AS data_mais_recente
FROM bronze.tb_orders;

data_mais_antiga,data_mais_recente
2016-09-04,2018-10-17


Criação da tabela bronze.tb_cotacao_dolar:

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

dbutils.widgets.text("data_inicio", "09-04-2016")
dbutils.widgets.text("data_fim", "10-17-2018")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = f"""
https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json
""".replace("\n", "")

response = requests.get(url)
response.raise_for_status()

dados = response.json().get("value", [])

df = spark.createDataFrame(dados)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.tb_cotacao_dolar")

display(df)

cotacaoCompra,dataHoraCotacao
3.2715,2016-09-05 13:09:55.659
3.2446,2016-09-06 13:02:39.984
3.1928,2016-09-08 13:03:53.968
3.2632,2016-09-09 13:14:00.885
3.2848,2016-09-12 13:08:01.541
3.2966,2016-09-13 13:03:56.534
3.3256,2016-09-14 13:05:51.819
3.332,2016-09-15 13:08:34.825
3.2998,2016-09-16 13:04:11.365
3.263,2016-09-19 13:07:42.039
